# Episode 01: Vectors, Matrices & The Rank of Information

> **Getting Comfortable Series — Track 2: LLM Fine-Tuning, PEFT & Alignment**  
> *"Don't simplify the topic. Simplify the path to understanding it."*

$$\textbf{Understand} \longrightarrow \text{Build} \longrightarrow \text{Break} \longrightarrow \text{Introduce Tools} \longrightarrow \text{Apply} \longrightarrow \text{Revisit}$$

---

### What You Will Master in this Notebook
Fine-tuning literature is full of phrases like *"the update matrix has a low intrinsic rank"* or *"LoRA factorizes weight matrices"*. But what does that actually mean under the hood?

In this notebook, we start from the **absolute mathematical ground floor**:
1. **Tokens as Vectors:** What a word embedding actually looks like in numbers.
2. **Linear Transformations:** What actually happens when a token meets a weight matrix ($y = x \cdot W^T$).
3. **The Intuition of Matrix Rank:** What "rank" actually measures (informational independence vs redundancy) using small $2 \times 2$ and $3 \times 3$ matrices you can calculate in your head.
4. **The Low-Rank Factorization Trick:** How a $3 \times 3$ matrix of 9 numbers can be represented by 6 numbers ($B \times A$).
5. **Scaling to LLM Dimensions:** How this simple matrix factorization saves millions of parameters in models like `Qwen` and `Llama`.


## 0. Setup & Deterministic Environment

We only need standard `numpy` and `torch`. If on a Mac, PyTorch will use CPU or Metal (MPS); every calculation in this notebook executes instantaneously.


In [ ]:
import numpy as np
import torch

torch.manual_seed(42)
np.random.seed(42)
print("Environment ready. PyTorch version:", torch.__version__)


---

## 1. Absolute Basics: A Token is Just a Vector

In a large language model, text is broken into tokens, and each token is converted into a vector of numbers called an **embedding**.

For example, suppose the word `"server"` is embedded into a tiny 3-dimensional vector:
$$x = \begin{bmatrix} 1.0 & 2.0 & 3.0 \end{bmatrix}$$

Each number represents a coordinate in semantic feature space.


In [ ]:
# 1 token, represented by 3 features
x = torch.tensor([[1.0, 2.0, 3.0]])
print("Token vector x shape:", x.shape)
print("Token vector x values:", x.numpy())


---

## 2. What Happens When a Token Meets a Weight Matrix?

Inside every attention head and MLP layer of an LLM, the token vector $x$ is transformed by multiplying against a pre-trained weight matrix $W_0$:

$$y = x \cdot W_0^T$$

Let's choose a simple $3 \times 3$ pre-trained weight matrix $W_0$:

$$W_0 = \begin{bmatrix} 1 & 0 & 2 \\ 0 & 3 & 1 \\ 2 & 1 & 0 \end{bmatrix}$$

Let's compute the output $y$ step-by-step:
- First element: $1(1) + 2(0) + 3(2) = 1 + 0 + 6 = 7$
- Second element: $1(0) + 2(3) + 3(1) = 0 + 6 + 3 = 9$
- Third element: $1(2) + 2(1) + 3(0) = 2 + 2 + 0 = 4$

Result: $y = \begin{bmatrix} 7 & 9 & 4 \end{bmatrix}$.


In [ ]:
W_0 = torch.tensor([
    [1.0, 0.0, 2.0],
    [0.0, 3.0, 1.0],
    [2.0, 1.0, 0.0]
])

# PyTorch linear projection
y = x @ W_0.T
print("Weight Matrix W_0:\n", W_0.numpy())
print("\nProjected Token Output y:\n", y.numpy())


---

## 3. What on Earth is Matrix "Rank"?

Now comes the fundamental concept behind all parameter-efficient fine-tuning (PEFT): **Rank**.

> **Definition:** The **Rank** of a matrix is the maximum number of **linearly independent** rows or columns it contains.  
> **In Plain English:** Rank is the **effective information dimension** of the matrix. It tells you how many genuinely unique patterns are stored inside.

Let's see this with two concrete $3 \times 3$ examples.

### Example A: Full Rank (Rank = 3)
Look at this identity matrix:
$$M_{\text{full}} = \begin{bmatrix} 1 & 0 & 0 \\ 0 & 1 & 0 \\ 0 & 0 & 1 \end{bmatrix}$$

Row 1 points strictly along the X-axis (`[1, 0, 0]`).  
Row 2 points strictly along the Y-axis (`[0, 1, 0]`).  
Row 3 points strictly along the Z-axis (`[0, 0, 1]`).  

None of these rows can be created by scaling or adding the other rows. All 3 directions are completely independent.  
Therefore, **Rank = 3 (Full Rank)**.

---

### Example B: Redundant / Low Rank (Rank = 1)
Now look at this matrix:

$$M_{\text{low}} = \begin{bmatrix} 1 & 2 & 3 \\ 2 & 4 & 6 \\ 3 & 6 & 9 \end{bmatrix}$$

Notice the pattern:
- **Row 2** is just **$2 \times \text{Row 1}$** ($[1, 2, 3] \times 2 = [2, 4, 6]$).
- **Row 3** is just **$3 \times \text{Row 1}$** ($[1, 2, 3] \times 3 = [3, 6, 9]$).

Even though there are **9 separate numbers** written in this matrix, there is really only **ONE unique direction of information** ($[1, 2, 3]$). The other two rows are completely redundant!  
Therefore, **Rank = 1**.


In [ ]:
M_full = torch.tensor([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0]
])

M_low = torch.tensor([
    [1.0, 2.0, 3.0],
    [2.0, 4.0, 6.0],
    [3.0, 6.0, 9.0]
])

rank_full = torch.linalg.matrix_rank(M_full).item()
rank_low  = torch.linalg.matrix_rank(M_low).item()

print(f"M_full rank: {rank_full} -> 3 independent directions of information.")
print(f"M_low rank:  {rank_low} -> Only 1 independent direction! (8 of the numbers are redundant copies)")


---

## 4. The Low-Rank Factorization Magic: $9$ numbers into $6$

Because $M_{\text{low}}$ has **Rank 1**, we do not need to store all 9 numbers!

Any rank-1 matrix of size $3 \times 3$ can be **decomposed (factorized)** into the outer product of two skinny vectors:
- A column vector $B$ of size $3 \times 1$
- A row vector $A$ of size $1 \times 3$

Let's test this arithmetic:

$$B = \begin{bmatrix} 1 \\ 2 \\ 3 \end{bmatrix}, \quad A = \begin{bmatrix} 1 & 2 & 3 \end{bmatrix}$$

Multiply $B \times A$:

$$B \times A = \begin{bmatrix} 1 \cdot 1 & 1 \cdot 2 & 1 \cdot 3 \\ 2 \cdot 1 & 2 \cdot 2 & 2 \cdot 3 \\ 3 \cdot 1 & 3 \cdot 2 & 3 \cdot 3 \end{bmatrix} = \begin{bmatrix} 1 & 2 & 3 \\ 2 & 4 & 6 \\ 3 & 6 & 9 \end{bmatrix}$$

Look at the storage comparison:
- Storing the full matrix: $3 \times 3 = 9$ numbers.
- Storing $B$ and $A$: $3 + 3 = \mathbf{6}$ numbers.
- **You saved $33\%$ of storage while reproducing the matrix perfectly.**


In [ ]:
B = torch.tensor([[1.0], [2.0], [3.0]])  # Shape: (3, 1)
A = torch.tensor([[1.0, 2.0, 3.0]])       # Shape: (1, 3)

# Matrix multiplication: (3, 1) @ (1, 3) -> (3, 3)
reconstructed = B @ A

print("B (3x1):\n", B.numpy())
print("\nA (1x3):\n", A.numpy())
print("\nProduct B @ A (3x3):\n", reconstructed.numpy())
print("\nExact match with M_low?", torch.equal(reconstructed, M_low))


---

## 5. Scaling to Real LLMs: Why Rank Saves Billion-Parameter Models

Now scale this exact insight up to modern transformer architectures like `Qwen2.5-7B` or `Llama-3-8B`:
- In these models, a single attention weight matrix has dimension:
  $$d_{\text{in}} = 4096, \quad d_{\text{out}} = 4096$$

### Full Matrix:
$$4096 \times 4096 = \mathbf{16,777,216 \text{ parameters}}$$
In standard 16-bit float, a single weight matrix takes **$33.5\text{ MB}$**.

### LoRA Factorization with Rank $r = 8$:
Instead of training a full $4096 \times 4096$ update matrix $\Delta W$, we decompose it into:
- $A \in \mathbb{R}^{8 \times 4096} = 32,768$ parameters
- $B \in \mathbb{R}^{4096 \times 8} = 32,768$ parameters
- **Total Parameters:** $32,768 + 32,768 = \mathbf{65,536 \text{ parameters}}$

$$\frac{65,536}{16,777,216} = 0.0039 \implies \mathbf{99.61\% \text{ reduction in parameters!}}$$


In [ ]:
def print_rank_scaling_table(d_in=4096, d_out=4096):
    full_params = d_in * d_out
    full_mb = (full_params * 2) / (1024**2) # FP16
    
    print(f"Base Matrix Dimension: {d_in} x {d_out}")
    print(f"Full Update Params:    {full_params:,} ({full_mb:.2f} MB)\n")
    print(f"{'Rank (r)':<10} | {'LoRA Params':<15} | {'Memory (FP16)':<15} | {'Savings %':<10}")
    print("-" * 60)
    
    for r in [1, 2, 4, 8, 16, 32, 64]:
        lora_params = (d_in * r) + (r * d_out)
        lora_mb = (lora_params * 2) / (1024**2)
        savings = (1 - lora_params / full_params) * 100
        print(f"{r:<10} | {lora_params:<15,} | {lora_mb:<12.3f} MB | {savings:.2f}%")

print_rank_scaling_table(4096, 4096)


---

## 🎯 Episode 01 Comfort Recap

1. **A token is a vector** representing coordinates in semantic feature space.
2. **A linear layer** computes $y = x W^T$, transforming token coordinates into new representations.
3. **Matrix Rank** measures the number of genuinely independent directions of information.
4. **Low-Rank Factorization ($B \times A$):** If an update $\Delta W$ has a low intrinsic rank, we can decompose a large $d \times k$ matrix into two skinny matrices $B (d \times r)$ and $A (r \times k)$.
5. **Why this matters for LLMs:** Adapting an LLM for formatting, tone, or tool calling only requires rank $r=8$ or $r=16$, slashing trainable parameters by **$99.6\%$**.

---

### Up Next:
In **Episode 02: Full Fine-Tuning (FFT) vs PEFT & Memory Math**, we explore what happens when we *don't* use low rank:
- The 2D Orthogonal Grid (Objective vs Weight Update).
- Why FFT requires $16\text{ bytes}$ per parameter (exploding an 8B model into $96+\text{ GB}$ VRAM).
- The exact hardware math formula you need before launching any training run.
